# Point Cloud Fusion with LoFTR Frame Alignment

Requires [LoFTR](https://github.com/zju3dv/LoFTR): `pip install git+https://github.com/zju3dv/LoFTR.git`

## Depth estimation + LoFTR-based frame alignment

Runs MiDaS depth estimation per frame, then uses LoFTR keypoint matching between consecutive frames to estimate camera translation and chain the point clouds into a shared frame.

In [ ]:
import torch
import cv2
import numpy as np
import open3d as o3d
import glob
from kornia.feature import LoFTR
#from src.loftr import default_cfg
from kornia.utils import image_to_tensor, tensor_to_image

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas = torch.hub.load("isl-org/MiDaS", "DPT_Hybrid").to(device).eval()
midas_transform = torch.hub.load("isl-org/MiDaS", "transforms").dpt_transform
loftr = LoFTR().to(device).eval()

image_paths = ['./combine-2/caml/test2/picture_cam_l20_s.jpg', './combine-2/camr/test2/picture_cam_r19_s.jpg'] #sorted(glob.glob("cam-mpu/picture_test_ai_with_oreient*.jpg"))
print(f"🖼️ Total images: {len(image_paths)}")

pcd_all = []
last_translation = np.zeros(3)

for idx, path in enumerate(image_paths):
    img = cv2.imread(path)
    if img is None:
        print(f"❌ Could not load image: {path}")
        continue
    # if np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)) > 240:
    #     continue
    img = cv2.resize(img, (384, 256))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Depth Estimation
    input_tensor = midas_transform(img_rgb).to(device)
    with torch.no_grad():
        depth = midas(input_tensor)
        depth = torch.nn.functional.interpolate(depth.unsqueeze(1), size=img.shape[:2], mode="bilinear", align_corners=False).squeeze().cpu().numpy()

    # Generate point cloud
    h, w = depth.shape
    fx = fy = w / 2
    cx, cy = w / 2, h / 2

    points, colors = [], []
    for y in range(h):
        for x in range(w):
            z = depth[y, x]
            if z <= 0: continue
            X = (x - cx) * z / fx
            Y = (y - cy) * z / fy
            pt = np.array([X, Y, z]) + last_translation
            points.append(pt)
            colors.append(img[y, x] / 255.0)

    # Add current cloud
    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(np.array(points))
    cloud.colors = o3d.utility.Vector3dVector(np.array(colors))
    pcd_all.append(cloud)

    print(f"🖼️ Processed image {idx + 1}/{len(image_paths)}: {path} - Points: {len(points)}")

    # Use LoFTR to compute shift between current and next image
    if idx + 1 < len(image_paths):
        img1 = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img2 = cv2.imread(image_paths[idx + 1], cv2.IMREAD_GRAYSCALE)
        if img2 is None:
            print(f"❌ Could not load image: {image_paths[idx + 1]}")
            continue
        img2 = cv2.resize(img2, (384, 256))
        t1 = image_to_tensor(img1 / 255.0).float()[None].to(device)
        t2 = image_to_tensor(img2 / 255.0).float()[None].to(device)

        with torch.no_grad():
            data = {'image0': t1, 'image1': t2}
            loftr(data)
            if 'keypoints0' in data and 'keypoints1' in data and data['keypoints0'].numel() > 0 and data['keypoints1'].numel() > 0:
                flow = data['keypoints1'] - data['keypoints0']
                dx, dy = flow[:, 0].mean().item(), flow[:, 1].mean().item()
                last_translation += np.array([dx / 100, dy / 100, 0])  # scale is empirical


## Refine alignment with ICP registration

Downsamples both clouds, does a rough RANSAC-based global registration, then refines with point-to-plane ICP before merging.

In [ ]:
import open3d as o3d
import numpy as np

# Load or define your left and right point clouds
pcd_left = pcd_all[0]
pcd_right = pcd_all[1]

# Step 1: Preprocess (downsample, estimate normals)
def preprocess(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100)
    )
    return pcd_down, fpfh

voxel_size = 0.01  # Adjust depending on scale
left_down, left_fpfh = preprocess(pcd_left, voxel_size)
right_down, right_fpfh = preprocess(pcd_right, voxel_size)

# Step 2: Global registration (rough alignment)
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source=right_down,
    target=left_down,
    source_feature=right_fpfh,
    target_feature=left_fpfh,
    mutual_filter=True,  # 🔧 Required argument
    max_correspondence_distance=voxel_size * 1.5,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size * 1.5)
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 1000)
)


# Step 3: ICP refinement (precise alignment)
result_icp = o3d.pipelines.registration.registration_icp(
    right_down, left_down, max_correspondence_distance=voxel_size * 0.4,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

# Step 4: Transform right cloud and merge
pcd_right.transform(result_icp.transformation)

merged = pcd_left + pcd_right
merged_down = merged.voxel_down_sample(voxel_size)

# Save and visualize
o3d.io.write_point_cloud("fused_registered_output.ply", merged_down)
o3d.visualization.draw_geometries([merged_down])


## View the fused result

In [ ]:
import open3d as o3d

# Load .ply file
pcd = o3d.io.read_point_cloud("midas_loftr_fused.ply")

# View summary
print(pcd)
print("Number of points:", len(pcd.points))

# Visualize it in a separate interactive window
o3d.visualization.draw_geometries([pcd])
